<a href="https://colab.research.google.com/github/jcmachicao/knowledge_engineering/blob/main/U3__simulador_neural_theorem_prover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Neural Theorem Prover (NTP) Simplificado**

Combina:
  - Backward chaining (como Prolog)
  - Embeddings vectoriales entrenables para predicados/entidades
  - Unificación suave (soft unification) diferenciable
  - Entrenamiento con PyTorch


## La idea central del NTP:

**La clave conceptual**:

El sistema puede aprender que padre y progenitor son similares sin que nadie se lo diga explícitamente, porque sus embeddings convergen durante el entrenamiento. Eso es lo que distingue al NTP de un Prolog puro.

## Arquitectura del código:

Prueba metas lógicas como Prolog (simbólico)

* Los embeddings de los predicados son vectores entrenables
* El "éxito" de una unificación es diferenciable (similitud vectorial)

Lógica:

* Representación lógica — Term, Atom, Rule implementan cláusulas de Horn clásicas (como Prolog)

* EmbeddingStore — cada predicado y entidad tiene un vector entrenable. La similitud coseno entre dos símbolos da el score de unificación (diferenciable)

* Soft Unification — en vez de unificación binaria (éxito/fallo), produce un score ∈ [0,1]. Predicados parecidos unifican con score alto, no solo predicados idénticos

* Backward chaining neural — recursión igual que Prolog, pero con scores:

* * OR entre reglas → max(scores)
* * AND entre submetas → product(scores)
* Entrenamiento — los embeddings se ajustan con log-loss para que ejemplos positivos → score 1 y negativos → score 0



## Simulador

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from itertools import product

In [ ]:
# ─────────────────────────────────────────────
# 1. REPRESENTACIÓN DE TÉRMINOS LÓGICOS
# ─────────────────────────────────────────────

class Term:
    """Un término puede ser constante, variable o predicado aplicado."""
    def __init__(self, name, is_var=False):
        self.name = name
        self.is_var = is_var  # Variables empiezan con mayúscula por convención

    def __repr__(self):
        return f"?{self.name}" if self.is_var else self.name

class Atom:
    """Un átomo: predicado(arg1, arg2, ...)"""
    def __init__(self, predicate, args):
        self.predicate = predicate  # str
        self.args = args            # list[Term]

    def __repr__(self):
        args_str = ", ".join(str(a) for a in self.args)
        return f"{self.predicate}({args_str})"

class Rule:
    """
    Cláusula de Horn: head :- body
    Si body está vacío, es un hecho.
    """
    def __init__(self, head: Atom, body: list = None):
        self.head = head
        self.body = body or []

    def __repr__(self):
        if not self.body:
            return f"{self.head}."
        body_str = ", ".join(str(b) for b in self.body)
        return f"{self.head} :- {body_str}."

In [ ]:
# ─────────────────────────────────────────────
# 2. EMBEDDINGS NEURALES
# ─────────────────────────────────────────────

class EmbeddingStore(nn.Module):
    """
    Almacena embeddings entrenables para predicados y entidades.
    La similitud entre embeddings = éxito de la unificación suave.
    """
    def __init__(self, symbols: list, dim: int = 16):
        super().__init__()
        self.dim = dim
        self.symbol_to_idx = {s: i for i, s in enumerate(symbols)}
        self.embeddings = nn.Embedding(len(symbols), dim)
        nn.init.uniform_(self.embeddings.weight, -0.5, 0.5)

    def get(self, symbol: str) -> torch.Tensor:
        if symbol not in self.symbol_to_idx:
            # Símbolo desconocido → vector aleatorio fijo
            return torch.randn(self.dim) * 0.1
        idx = torch.tensor(self.symbol_to_idx[symbol])
        return self.embeddings(idx)

    def similarity(self, sym_a: str, sym_b: str) -> torch.Tensor:
        """Similitud coseno entre dos símbolos → score de unificación [0,1]."""
        ea = self.get(sym_a)
        eb = self.get(sym_b)
        return (F.cosine_similarity(ea.unsqueeze(0), eb.unsqueeze(0)) + 1) / 2

In [ ]:
# ─────────────────────────────────────────────
# 3. UNIFICACIÓN SUAVE (SOFT UNIFICATION)
# ─────────────────────────────────────────────

def soft_unify_terms(t1: Term, t2: Term, emb_store: EmbeddingStore,
                     substitution: dict) -> tuple:
    """
    Intenta unificar dos términos.
    Retorna (score, nueva_substitución).
    score=1.0 → unificación perfecta
    score~0   → unificación fallida
    """
    # Resolver variables en la substitución
    t1 = substitution.get(t1.name, t1) if t1.is_var else t1
    t2 = substitution.get(t2.name, t2) if t2.is_var else t2

    if t1.is_var:
        new_sub = {**substitution, t1.name: t2}
        return torch.tensor(1.0), new_sub
    if t2.is_var:
        new_sub = {**substitution, t2.name: t1}
        return torch.tensor(1.0), new_sub

    # Ambas son constantes: similitud neuronal
    score = emb_store.similarity(t1.name, t2.name)
    return score, substitution

def soft_unify_atoms(a1: Atom, a2: Atom, emb_store: EmbeddingStore,
                     substitution: dict) -> tuple:
    """
    Unifica dos átomos: primero predicados, luego argumentos.
    El score final es el producto (AND lógico suave).
    """
    if len(a1.args) != len(a2.args):
        return torch.tensor(0.0), substitution

    # Similitud de predicados
    pred_score = emb_store.similarity(a1.predicate, a2.predicate)
    total_score = pred_score
    current_sub = substitution.copy()

    for arg1, arg2 in zip(a1.args, a2.args):
        score, current_sub = soft_unify_terms(arg1, arg2, emb_store, current_sub)
        total_score = total_score * score  # AND suave

    return total_score, current_sub

In [ ]:
# ─────────────────────────────────────────────
# 4. BACKWARD CHAINING NEURAL
# ─────────────────────────────────────────────

def prove(goal: Atom, rules: list, emb_store: EmbeddingStore,
          substitution: dict = None, depth: int = 0, max_depth: int = 3) -> torch.Tensor:
    """
    Prueba recursiva de una meta usando backward chaining.
    Retorna el score máximo entre todas las ramas de prueba.

    OR  entre reglas que aplican  (max)
    AND entre submetas de una regla (product)
    """
    if substitution is None:
        substitution = {}

    if depth > max_depth:
        return torch.tensor(0.0)

    best_score = torch.tensor(0.0)

    for rule in rules:
        # Renombrar variables de la regla para evitar colisiones
        suffix = f"_{depth}_{id(rule)}"
        renamed_head = rename_vars(rule.head, suffix)
        renamed_body = [rename_vars(b, suffix) for b in rule.body]

        # Intentar unificar la meta con la cabeza de la regla
        unify_score, new_sub = soft_unify_atoms(goal, renamed_head, emb_store, substitution)

        if unify_score.item() < 0.01:  # Poda: si la unificación falla casi completamente
            continue

        if not renamed_body:
            # Es un hecho → score = score de unificación
            score = unify_score
        else:
            # Probar cada submeta del cuerpo (AND → producto)
            body_score = unify_score
            for subgoal in renamed_body:
                sg_score = prove(subgoal, rules, emb_store, new_sub, depth + 1, max_depth)
                body_score = body_score * sg_score

            score = body_score

        # OR entre ramas → máximo (con gradiente)
        best_score = torch.max(best_score, score)

    return best_score


def rename_vars(atom: Atom, suffix: str) -> Atom:
    """Renombra variables en un átomo para evitar colisiones entre reglas."""
    new_args = []
    for arg in atom.args:
        if arg.is_var:
            new_args.append(Term(arg.name + suffix, is_var=True))
        else:
            new_args.append(arg)
    return Atom(atom.predicate, new_args)

In [ ]:
# ─────────────────────────────────────────────
# 5. ENTRENAMIENTO
# ─────────────────────────────────────────────

def train_ntp(pos_examples: list, neg_examples: list, rules: list,
              emb_store: EmbeddingStore, epochs: int = 100, lr: float = 0.01):
    """
    Entrena los embeddings para que:
    - Ejemplos positivos → score alto
    - Ejemplos negativos → score bajo
    """
    optimizer = torch.optim.Adam(emb_store.parameters(), lr=lr)

    for epoch in range(epochs):
        optimizer.zero_grad()
        total_loss = torch.tensor(0.0)

        # Pérdida en ejemplos positivos: queremos score → 1
        for goal in pos_examples:
            score = prove(goal, rules, emb_store)
            loss = -torch.log(score + 1e-6)  # log-loss
            total_loss = total_loss + loss

        # Pérdida en ejemplos negativos: queremos score → 0
        for goal in neg_examples:
            score = prove(goal, rules, emb_store)
            loss = -torch.log(1 - score + 1e-6)
            total_loss = total_loss + loss

        total_loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"Epoch {epoch:3d} | Loss: {total_loss.item():.4f}")

In [ ]:
# ─────────────────────────────────────────────
# 6. DEMO
# ─────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 55)
    print("  Neural Theorem Prover — Demo")
    print("=" * 55)

    # ── Dominio: relaciones familiares ──────────────────────

    # Variables
    X = Term("X", is_var=True)
    Y = Term("Y", is_var=True)
    Z = Term("Z", is_var=True)

    # Constantes (personas)
    ana   = Term("ana")
    luis  = Term("luis")
    mia   = Term("mia")
    pedro = Term("pedro")
    sara  = Term("sara")

    # ── Base de conocimiento (reglas y hechos) ──────────────

    rules = [
        # Hechos: padre(padre, hijo)
        Rule(Atom("padre", [luis,  ana])),
        Rule(Atom("padre", [luis,  mia])),
        Rule(Atom("padre", [pedro, luis])),

        # Hechos: madre(madre, hijo)
        Rule(Atom("madre", [sara, ana])),
        Rule(Atom("madre", [sara, mia])),

        # Regla: abuelo(X, Z) :- padre(X, Y), padre(Y, Z)
        Rule(
            head=Atom("abuelo", [X, Z]),
            body=[Atom("padre", [X, Y]), Atom("padre", [Y, Z])]
        ),

        # Regla: hermano(X, Y) :- padre(Z, X), padre(Z, Y)
        Rule(
            head=Atom("hermano", [X, Y]),
            body=[Atom("padre", [Z, X]), Atom("padre", [Z, Y])]
        ),
    ]

    # ── Todos los símbolos del dominio ──────────────────────
    symbols = ["padre", "madre", "abuelo", "hermano",
               "ana", "luis", "mia", "pedro", "sara"]

    emb_store = EmbeddingStore(symbols, dim=16)

    # ── Prueba antes de entrenar ────────────────────────────
    print("\n[Antes del entrenamiento]")
    goal_pos = Atom("abuelo", [pedro, ana])
    goal_neg = Atom("abuelo", [ana, pedro])

    with torch.no_grad():
        s1 = prove(goal_pos, rules, emb_store)
        s2 = prove(goal_neg, rules, emb_store)
    print(f"  abuelo(pedro, ana)  → score: {s1.item():.4f}  [debería ser ~1]")
    print(f"  abuelo(ana, pedro)  → score: {s2.item():.4f}  [debería ser ~0]")

    # ── Entrenamiento ───────────────────────────────────────
    pos_examples = [
        Atom("padre",   [luis,  ana]),
        Atom("padre",   [luis,  mia]),
        Atom("padre",   [pedro, luis]),
        Atom("madre",   [sara,  ana]),
        Atom("abuelo",  [pedro, ana]),
        Atom("abuelo",  [pedro, mia]),
        Atom("hermano", [ana,   mia]),
    ]

    neg_examples = [
        Atom("padre",  [ana,   pedro]),
        Atom("abuelo", [ana,   pedro]),
        Atom("madre",  [luis,  pedro]),
    ]

    print("\n[Entrenamiento]")
    train_ntp(pos_examples, neg_examples, rules, emb_store, epochs=100, lr=0.05)

    # ── Prueba después de entrenar ──────────────────────────
    print("\n[Después del entrenamiento]")
    queries = [
        (Atom("abuelo",  [pedro, ana]),  True),
        (Atom("abuelo",  [pedro, mia]),  True),
        (Atom("hermano", [ana,   mia]),  True),
        (Atom("padre",   [luis,  ana]),  True),
        (Atom("abuelo",  [ana,   pedro]), False),
        (Atom("padre",   [ana,   pedro]), False),
    ]

    print(f"\n  {'Query':<30} {'Score':>8}  {'Esperado':>10}  {'OK?':>5}")
    print("  " + "-" * 60)
    with torch.no_grad():
        for goal, expected in queries:
            score = prove(goal, rules, emb_store).item()
            ok = "✓" if (score > 0.5) == expected else "✗"
            print(f"  {str(goal):<30} {score:>8.4f}  {'TRUE' if expected else 'FALSE':>10}  {ok:>5}")

    print("\n[Similitudes aprendidas entre predicados]")
    pairs = [("padre", "madre"), ("padre", "abuelo"), ("abuelo", "hermano")]
    with torch.no_grad():
        for a, b in pairs:
            sim = emb_store.similarity(a, b).item()
            print(f"  sim({a}, {b}) = {sim:.4f}")

    print("\nListo. Los embeddings han aprendido a representar")
    print("las relaciones para que el prover funcione mejor.")

  Neural Theorem Prover — Demo

[Antes del entrenamiento]
  abuelo(pedro, ana)  → score: 1.0000  [debería ser ~1]
  abuelo(ana, pedro)  → score: 0.3847  [debería ser ~0]

[Entrenamiento]
Epoch   0 | Loss: 1.2963
Epoch  20 | Loss: 0.1984
Epoch  40 | Loss: 0.0367
Epoch  60 | Loss: 0.0205
Epoch  80 | Loss: -0.0000

[Después del entrenamiento]

  Query                             Score    Esperado    OK?
  ------------------------------------------------------------
  abuelo(pedro, ana)               1.0000        TRUE      ✓
  abuelo(pedro, mia)               1.0000        TRUE      ✓
  hermano(ana, mia)                1.0000        TRUE      ✓
  padre(luis, ana)                 1.0000        TRUE      ✓
  abuelo(ana, pedro)               0.0000       FALSE      ✓
  padre(ana, pedro)                0.0000       FALSE      ✓

[Similitudes aprendidas entre predicados]
  sim(padre, madre) = 0.1081
  sim(padre, abuelo) = 0.8796
  sim(abuelo, hermano) = 0.5537

Listo. Los embeddings han aprend

Referencia conceptual:
* Rocktäschel & Riedel (2017) "End-to-end Differentiable Proving"